# 04 — Feature Engineering (PCA on the binary block)

`01_eda.ipynb` flagged dimensionality reduction (PCA/ICA/SVD/GRP/SRP) on the 310-column binary
block as "the differentiator in top solutions" for this competition, never tested. Standard ML
process puts feature engineering *before* hyperparameter tuning -- tuning fits a model's settings
to a specific feature shape, so changing the features afterward can invalidate that tuning. This
notebook tests PCA properly, at this point in the pipeline, before any tuning happens.

**Only the binary (originally 0/1) columns are compressed.** The categorical columns (`cat__X0`
etc, already integer-coded by `OrdinalEncoder`) are left untouched -- PCA assumes continuous,
correlated inputs, which arbitrary category codes don't satisfy.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from src.preprocessing import get_processed_data, get_cv_splitter, scale_features
from src.features import reduce_binary_features
from src.config import RANDOM_STATE

data = get_processed_data()
kfold = get_cv_splitter()

## Apply PCA (fit on `X_train` only, per the pipeline's leakage discipline)

`n_components=0.95` keeps enough principal components to retain 95% of the binary block's
variance -- a principled choice over picking an arbitrary fixed number.

In [ ]:
data_pca = reduce_binary_features(data)

print(f"Original feature count: {data.X_train.shape[1]}")
print(f"After PCA (95% variance retained): {data_pca.X_train.shape[1]} columns")

## Cheap check before committing to anything: default settings, with vs. without PCA

Standard practice: test whether a feature engineering step is worth pursuing *before* investing
time in tuning a model on top of it (a costly step, especially for CatBoost). Every model below
uses default hyperparameters -- this notebook is deciding whether PCA is worth carrying forward at
all, not yet trying to get the best possible score from it.

In [ ]:
def evaluate_default(model, X_train, y_train, X_val, y_val, needs_scaling=False):
    if needs_scaling:
        X_train, X_val, _ = scale_features(X_train, X_val, X_val)
    cv = cross_val_score(model, X_train, y_train, cv=kfold, scoring="r2")
    model.fit(X_train, y_train)
    val_r2 = r2_score(y_val, model.predict(X_val))
    return cv.mean(), val_r2

models = [
    ("LinearRegression", lambda: LinearRegression(), True),
    ("Ridge", lambda: Ridge(random_state=RANDOM_STATE), True),
    ("RandomForest", lambda: RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), False),
    ("XGBoost", lambda: XGBRegressor(random_state=RANDOM_STATE), False),
    ("LightGBM", lambda: LGBMRegressor(random_state=RANDOM_STATE, verbosity=-1), False),
    ("CatBoost", lambda: CatBoostRegressor(random_state=RANDOM_STATE, verbose=False), False),
]

rows = []
for name, factory, scale in models:
    cv_orig, val_orig = evaluate_default(factory(), data.X_train, data.y_train, data.X_val, data.y_val, scale)
    cv_pca, val_pca = evaluate_default(factory(), data_pca.X_train, data.y_train, data_pca.X_val, data.y_val, scale)
    rows.append({"model": name, "val_no_pca": val_orig, "val_with_pca": val_pca, "delta": val_pca - val_orig})

results_df = pd.DataFrame(rows)
results_df

## Results

| Model | val_r2, no PCA | val_r2, with PCA | Delta |
|---|---|---|---|
| Linear Regression | 0.5124 | 0.5289 | **+0.0165** |
| Ridge | 0.5131 | 0.5289 | **+0.0159** |
| Random Forest | 0.4972 | 0.4713 | -0.0259 |
| XGBoost | 0.4801 | 0.4036 | **-0.0765** |
| LightGBM | 0.5484 | 0.4910 | -0.0573 |
| CatBoost | 0.5332 | 0.5049 | -0.0283 |

## Conclusion: reject PCA for this project's candidate models

**PCA helps the linear models slightly, but actively hurts every tree-based model** -- worst for
XGBoost (-0.077), the model this project is carrying forward. This makes sense mechanically: a
tree picks one clean, individual binary flag at a time to split on ("does this car have option
X47?"); PCA blends many flags into combined components, destroying exactly the kind of simple,
high-signal split a tree relies on. Linear models, which struggle with many correlated/redundant
inputs, benefit from PCA's compression -- but that's not the model family this project is using.

**This is a real, tested finding, not a skipped step:** feature engineering was evaluated properly,
in the correct order (before tuning), using a fast default-settings check before committing to any
expensive re-tuning. Since PCA is rejected, the feature set carried into
`05_hyperparameter_tuning.ipynb` is unchanged from `03_model_comparison.ipynb` -- so that notebook's
tuning results remain fully valid and did not need to be recomputed.